# 04 - Review errors and reload a prediction
**Question:** Where is the forecast weak, and can we reuse the saved model?

Run notebooks 01-03 first. This notebook reports monthly and commute-hour errors, plots observed rentals
against predictions, and scores a target-free historical example. This is an offline review, not live monitoring.
Do not change the model using these test findings and still call the same test period untouched.


In [ ]:
from pathlib import Path
import sys
# Support opening the notebook from the project root or notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'mobility').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'mobility').is_dir():
    raise RuntimeError('Open Jupyter from the project folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from mobility.data import OUTPUT, invalidate_outputs
import pandas as pd
import numpy as np
import json
from IPython.display import display

from mobility.features import FEATURES
from mobility.evaluation import forecast_metrics
from mobility.scoring import predict_saved
invalidate_outputs(4)
test = pd.read_csv(OUTPUT / 'test.csv')
results = pd.read_csv(OUTPUT / 'test_predictions.csv')
results['month'] = pd.to_datetime(results.timestamp).dt.to_period('M').astype(str)
monthly = []
for month, group in results.groupby('month'):
    row = {'month': month, 'hours': len(group), **forecast_metrics(group.cnt, group.prediction)}
    monthly.append(row)
monthly = pd.DataFrame(monthly)
monthly.to_csv(OUTPUT / 'monthly_errors.csv', index=False)
display(monthly.round(2))


## 1. Inspect commute-hour errors
Our definition is working days at hours 7, 8, 9, 17, 18, and 19. This is a transparent analysis rule,
not a guarantee about all cities. Compare these errors with the overall test metrics in notebook 02.
Monthly changes can reflect seasonality, events, missing hours, or model error; this table alone cannot identify the cause.


In [ ]:
peak = results[(results.workingday == 1) & results.hr.isin([7, 8, 9, 17, 18, 19])]
print('Eligible commute hours:', len(peak))
display(pd.DataFrame([forecast_metrics(peak.cnt, peak.prediction)]).round(2))
import matplotlib.pyplot as plt
first_week = results[pd.to_datetime(results.timestamp) < pd.Timestamp('2012-10-08')]
ax = first_week.set_index(pd.to_datetime(first_week.timestamp))[['cnt', 'prediction']].plot(figsize=(10, 4))
ax.set(xlabel='Timestamp', ylabel='Rentals per observed hour', title='First test week: actual rentals and frozen forecast')
plt.tight_layout()
plt.savefig(OUTPUT / 'error_review.png', dpi=150)
plt.show()


## 2. Reload and score an example without its answer
We use the first eligible test row as a historical demonstration. The input contains the eight
predictors only: no `cnt`, `casual`, `registered`, or observed future weather. Its two lag counts were
already recorded before this hour. This is not a forecast for today's bike-sharing system.

Reloaded predictions must match the selected prediction saved by notebook 02 within numerical tolerance.


In [ ]:
example = test[FEATURES].head(1).copy()
example.to_csv(OUTPUT / 'example_input.csv', index=False)
reloaded = predict_saved(example)
np.testing.assert_allclose(reloaded, results.prediction.head(1).to_numpy(), rtol=1e-10, atol=1e-10)
scored = example.copy()
scored['predicted_rentals'] = reloaded
scored.to_csv(OUTPUT / 'example_prediction.csv', index=False)
display(scored.round(2))
print('Saved model reproduces the original forecast.')


## Explain this project in an interview
"I used hourly public bike-sharing records to compare a same-hour-last-week rule with gradient boosting.
I built exact-time historical features, kept later dates separate for validation and test, and selected
using validation MAE. I then compared hypothetical capacity buffers and reviewed where the model made errors."

Replace general statements with the actual results from `output/metrics.json`. Be ready to explain a
single row's two lags, why the target components are excluded, and why the capacity exercise is only a proxy.

**Limitations:** one historical city/system; missing clock hours; evaluation limited to rows with both lags;
no weather forecasts; no station inventory; no deployed API; no measured financial or service benefit.
Calendar year supports this 2011-2012 exercise, not general future-year extrapolation.

**Finished:** the project can be rerun from notebook 01. See the local guides for a slower walkthrough
and interview questions. Rerun later notebooks whenever an earlier stage changes.
